# 🔍 Fraud Detection Model Trainer
**Chạy từng cell theo thứ tự.**  
Kết quả cuối: tải về file `models.zip` → giải nén vào folder `models/` cạnh `main.py`

## Cell 1 — Cài thư viện

In [ ]:
!pip install -q xgboost scikit-learn pandas numpy
print('✅ Cài thư viện xong')

## Cell 2 — Upload `fraudTrain.csv`
> File ~500MB, chờ upload xong mới chạy cell tiếp theo

In [ ]:
from google.colab import files
uploaded = files.upload()   # Bấm 'Choose Files' → chọn fraudTrain.csv
print('✅ Upload xong:', list(uploaded.keys()))

## Cell 3 — Import thư viện

In [ ]:
import os
import pandas as pd
import numpy as np
import pickle
import zipfile
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, recall_score
from xgboost import XGBClassifier
print('✅ Import xong')

## Cell 4 — Định nghĩa hàm Feature Engineering
> Copy từ `feature_engineering.py` — đã fix `dob` đa format Kaggle

In [ ]:
# ── Haversine ────────────────────────────────────────────────
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

# ── Parse dob đa format (fix Kaggle M/d/yyyy) ────────────────
REF_DATE    = pd.Timestamp('2020-01-01')
DOB_FORMATS = ['%Y-%m-%d', '%m/%d/%Y', '%m/%d/%y', '%d/%m/%Y']

def _parse_dob_series(dob_series, ref):
    parsed = pd.Series(pd.NaT, index=dob_series.index)
    for fmt in DOB_FORMATS:
        mask = parsed.isna()
        if not mask.any():
            break
        try:
            attempt = pd.to_datetime(dob_series[mask], format=fmt, errors='coerce')
            parsed[mask] = attempt
        except Exception:
            continue
    age = (ref - parsed).dt.days / 365.25
    age = age.where((age > 0) & (age < 120), other=np.nan)
    return age.fillna(40.0)

# ── build_features ───────────────────────────────────────────
def build_features(df, encoders=None, is_training=False):
    df_feat = pd.DataFrame(index=df.index)

    # Time
    if 'unix_time' in df.columns:
        dt = pd.to_datetime(df['unix_time'], unit='s')
    elif 'trans_date_trans_time' in df.columns:
        dt = pd.to_datetime(df['trans_date_trans_time'])
    else:
        dt = pd.Series(pd.Timestamp('now'), index=df.index)

    df_feat['hour']        = dt.dt.hour
    df_feat['day_of_week'] = dt.dt.dayofweek
    df_feat['is_weekend']  = df_feat['day_of_week'].isin([5, 6]).astype(int)
    df_feat['is_night']    = ((df_feat['hour'] >= 2) & (df_feat['hour'] <= 5)).astype(int)

    # Amount
    df_feat['amt']     = df['amt'] if 'amt' in df.columns else 0.0
    df_feat['amt_log'] = np.log1p(df_feat['amt'])

    # Distance
    if all(c in df.columns for c in ['lat', 'long', 'merch_lat', 'merch_long']):
        df_feat['distance_km'] = haversine(
            df['lat'].values, df['long'].values,
            df['merch_lat'].values, df['merch_long'].values
        )
    else:
        df_feat['distance_km'] = 0.0
    df_feat['distance_log'] = np.log1p(df_feat['distance_km'])

    # Age — dùng REF_DATE cố định khi training
    if 'dob' in df.columns:
        df_feat['age'] = _parse_dob_series(df['dob'].astype(str), REF_DATE)
    else:
        df_feat['age'] = 40.0

    # City pop
    df_feat['city_pop_log'] = np.log1p(df['city_pop']) if 'city_pop' in df.columns else 10.0

    # Categorical encoding
    categories = ['category', 'merchant']
    if is_training:
        encoders = {}
        for col in categories:
            le = LabelEncoder()
            series = df[col].astype(str) if col in df.columns else pd.Series(['UNKNOWN'] * len(df))
            df_feat[col + '_encoded'] = le.fit_transform(series)
            classes = list(le.classes_)
            if 'UNKNOWN' not in classes:
                classes.append('UNKNOWN')
            le.classes_ = np.array(classes)
            encoders[col] = le
        return df_feat, encoders
    else:
        for col in categories:
            if encoders and col in encoders:
                le = encoders[col]
                if col in df.columns:
                    series = df[col].astype(str).values
                    mapping = {cls: idx for idx, cls in enumerate(le.classes_)}
                    unknown_idx = mapping.get('UNKNOWN', len(le.classes_) - 1)
                    df_feat[col + '_encoded'] = [mapping.get(x, unknown_idx) for x in series]
                else:
                    df_feat[col + '_encoded'] = len(le.classes_) - 1
            else:
                df_feat[col + '_encoded'] = 0
        return df_feat

print('✅ Định nghĩa hàm xong')

## Cell 5 — Load data & build features

In [ ]:
print('📂 Đọc fraudTrain.csv...')
train_df = pd.read_csv('fraudTrain.csv')
print(f'   Tổng: {len(train_df):,} dòng | Fraud: {train_df["is_fraud"].sum():,} ({train_df["is_fraud"].mean()*100:.2f}%)')
print(f'   Cột: {list(train_df.columns)}')

# Dùng 20% cuối làm test (giữ thứ tự thời gian)
split    = int(len(train_df) * 0.8)
test_df  = train_df.iloc[split:].copy()
train_df = train_df.iloc[:split].copy()
print(f'\n   → Train: {len(train_df):,} | Test: {len(test_df):,}')

print('\n🔧 Building features...')
X_train, encoders = build_features(train_df, is_training=True)
y_train = train_df['is_fraud']
X_test  = build_features(test_df, encoders=encoders, is_training=False)
y_test  = test_df['is_fraud']

print(f'   Features ({X_train.shape[1]} cột): {list(X_train.columns)}')
print(f'   X_train: {X_train.shape} | X_test: {X_test.shape}')

# Kiểm tra age đã parse đúng chưa
age_sample = X_train['age'].describe()
print(f'\n   Kiểm tra age (nếu mean ~ 40 là vẫn lỗi, phải đa dạng hơn):')
print(age_sample)

## Cell 6 — Train RandomForest

In [ ]:
print('🌲 Training RandomForest (200 cây)...')
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1          # dùng hết CPU Colab
)
rf_model.fit(X_train, y_train)
print('✅ RandomForest xong')

## Cell 7 — Train XGBoost

In [ ]:
print('⚡ Training XGBoost (300 cây)...')
xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    random_state=42,
    eval_metric='logloss',
    n_jobs=-1,
    tree_method='hist'  # nhanh hơn trên Colab
)
xgb_model.fit(X_train, y_train)
print('✅ XGBoost xong')

## Cell 8 — Train IsolationForest

In [ ]:
# contamination tính từ tỷ lệ fraud thực tế
fraud_rate    = float(y_train.mean())
contamination = float(np.clip(fraud_rate, 0.001, 0.1))
print(f'🌲 Training IsolationForest (contamination={contamination:.4f} từ fraud_rate={fraud_rate:.4f})...')
iso_model = IsolationForest(contamination=contamination, random_state=42)
iso_model.fit(X_train)
print('✅ IsolationForest xong')

## Cell 9 — Đánh giá trên test set

In [ ]:
print('📊 Đánh giá ensemble trên test set...\n')
rf_preds      = rf_model.predict_proba(X_test)[:, 1]
xgb_preds     = xgb_model.predict_proba(X_test)[:, 1]
iso_preds_raw = iso_model.predict(X_test)
iso_probs     = np.where(iso_preds_raw == -1, 1.0, 0.0)

fraud_score   = 0.4 * rf_preds + 0.4 * xgb_preds + 0.2 * iso_probs
ensemble_pred = (fraud_score > 0.5).astype(int)

print(classification_report(y_test, ensemble_pred, target_names=['Bình thường', 'Fraud']))
print('Confusion Matrix:')
print(confusion_matrix(y_test, ensemble_pred))
recall = recall_score(y_test, ensemble_pred)
print(f'\nRecall (fraud): {recall:.4f}')

## Cell 10 — Lưu models & đóng gói ZIP

In [ ]:
print('💾 Lưu models...')
os.makedirs('models', exist_ok=True)
with open('models/rf_model.pkl',  'wb') as f: pickle.dump(rf_model,  f)
with open('models/xgb_model.pkl', 'wb') as f: pickle.dump(xgb_model, f)
with open('models/iso_model.pkl', 'wb') as f: pickle.dump(iso_model, f)
with open('models/encoders.pkl',  'wb') as f: pickle.dump(encoders,  f)
print('   ✅ Lưu 4 file xong')

print('\n📦 Đóng gói models.zip...')
with zipfile.ZipFile('models.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in ['rf_model.pkl', 'xgb_model.pkl', 'iso_model.pkl', 'encoders.pkl']:
        zf.write(f'models/{fname}', fname)
size_mb = os.path.getsize('models.zip') / 1024 / 1024
print(f'   ✅ models.zip ({size_mb:.1f} MB) — sẵn sàng download')

## Cell 11 — Download `models.zip` về máy

In [ ]:
from google.colab import files
files.download('models.zip')
print('✅ Download xong!')
print()
print('Bước tiếp theo:')
print('  1. Giải nén models.zip → lấy 4 file .pkl')
print('  2. Đặt vào folder models/ cạnh main.py')
print('  3. Chạy: python main.py')